In this notebook, I will use the data generated in poseRecognition.ipynb to extract additional features. The primary focus will be on calculating the angles between joints and measuring the speed of movement.

In [ ]:
pip install pandas

In [ ]:
import pandas as pd


1. Load the existing data

In [ ]:
data = pd.read_csv('output\pose_2.csv')
data 

1.2 Data cleanup 

Since the data was generated using MediaPipe, some joints are not relevant for my analysis. The joints are organized as follows:
| ID  | Landmark           |
|-----|------------------|
| 0   | NOSE              |
| 1   | LEFT_EYE_INNER    |
| 2   | LEFT_EYE          |
| 3   | LEFT_EYE_OUTER    |
| 4   | RIGHT_EYE_INNER   |
| 5   | RIGHT_EYE         |
| 6   | RIGHT_EYE_OUTER   |
| 7   | LEFT_EAR          |
| 8   | RIGHT_EAR         |
| 9   | MOUTH_LEFT        |
| 10  | MOUTH_RIGHT       |
| 11  | LEFT_SHOULDER     |
| 12  | RIGHT_SHOULDER    |
| 13  | LEFT_ELBOW        |
| 14  | RIGHT_ELBOW       |
| 15  | LEFT_WRIST        |
| 16  | RIGHT_WRIST       |
| 17  | LEFT_PINKY        |
| 18  | RIGHT_PINKY       |
| 19  | LEFT_INDEX        |
| 20  | RIGHT_INDEX       |
| 21  | LEFT_THUMB        |
| 22  | RIGHT_THUMB       |
| 23  | LEFT_HIP          |
| 24  | RIGHT_HIP         |
| 25  | LEFT_KNEE         |
| 26  | RIGHT_KNEE        |
| 27  | LEFT_ANKLE        |
| 28  | RIGHT_ANKLE       |
| 29  | LEFT_HEEL         |
| 30  | RIGHT_HEEL        |
| 31  | LEFT_FOOT_INDEX   |
| 32  | RIGHT_FOOT_INDEX  |


I will exclude irrelevant joints, such as ears ore pinkys, from the analysis.


In [ ]:
joints = [
    (11, "LEFT_SHOULDER"),
    (12, "RIGHT_SHOULDER"),
    (13, "LEFT_ELBOW"),
    (14, "RIGHT_ELBOW"),
    (15, "LEFT_WRIST"),
    (16, "RIGHT_WRIST"),
    (23, "LEFT_HIP"),
    (24, "RIGHT_HIP"),
    (25, "LEFT_KNEE"),
    (26, "RIGHT_KNEE"),
    (27, "LEFT_ANKLE"),
    (28, "RIGHT_ANKLE"),
    (29, "LEFT_HEEL"),
    (30, "RIGHT_HEEL"),
    (31, "LEFT_FOOT_INDEX"),
    (32, "RIGHT_FOOT_INDEX")
]


In [ ]:
joints_ids = [id for id, name in joints]

data_filtered = data[data["joint"].isin(joints_ids)]
data_filtered


1.3 Adding Unknown Data Using Linear Interpolation

In [ ]:
if data_filtered.isna().any().any():
    print("there are NaN values in the data")
    data_filtered = data_filtered.interpolate(method="linear")


2. Compute joint angles


In [ ]:
import numpy as np

Pivot data to have joints as columns per frame
Columns will be like: joint0_x, joint0_y, joint0_z, joint1_x, ...

In [ ]:

pose_pivot = data_filtered.pivot(index='frame', columns='joint', values=['x','y','z'])
pose_pivot.columns = [f"{axis}{joint}" for axis, joint in pose_pivot.columns]


Calculate the angle at point b formed by points a-b-c in 3D.
Returns angle in degrees.


In [ ]:
def calculate_angle(a, b, c):

    ba = np.array(a) - np.array(b)
    bc = np.array(c) - np.array(b)
    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    cos_angle = np.clip(cos_angle, -1.0, 1.0)  # Numerical stability
    angle = np.arccos(cos_angle)
    return np.degrees(angle)


2.2 Now I have to add a joint tripplets to calculat the angles between them.

In [ ]:

angle_defs = {
    "elbow_left":  ("LEFT_SHOULDER", "LEFT_ELBOW", "LEFT_WRIST"),
    "elbow_right": ("RIGHT_SHOULDER", "RIGHT_ELBOW", "RIGHT_WRIST"),
    "knee_left":   ("LEFT_HIP", "LEFT_KNEE", "LEFT_ANKLE"),
    "knee_right":  ("RIGHT_HIP", "RIGHT_KNEE", "RIGHT_ANKLE"),
    "hip_left":    ("LEFT_SHOULDER", "LEFT_HIP", "LEFT_KNEE"),
    "hip_right":   ("RIGHT_SHOULDER", "RIGHT_HIP", "RIGHT_KNEE")
}

Adding the new features to the dataset

In [ ]:
for angle_name in angle_defs:
    pose_pivot[angle_name] = None

In [ ]:
for idx, row in pose_pivot.iterrows():
    coords = {
        name: (row[f'x{id}'], row[f'y{id}'], row[f'z{id}'])
        for id, name in joints
    }

    for angle_name, (a, b, c) in angle_defs.items():
        p1 = coords[a]
        p2 = coords[b]
        p3 = coords[c]
        pose_pivot.at[idx, angle_name] = calculate_angle(p1, p2, p3)

pose_pivot

2.3 relative distances between body parts

In [ ]:
distance_defs = {
    "foot_to_foot": ("LEFT_ANKLE", "RIGHT_ANKLE"),
    "foot_to_hip": ("LEFT_ANKLE", "LEFT_HIP", "RIGHT_ANKLE", "RIGHT_HIP"),
    "hip_to_shoulder": ("LEFT_HIP", "LEFT_SHOULDER", "RIGHT_HIP", "RIGHT_SHOULDER")
}

In [ ]:
def calculate_distance(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

In [ ]:
def calculate_distance4(a,b,c,d):
    left = calculate_distance(a,b)
    right = calculate_distance(c,d)
    return (left + right) / 2   

In [ ]:
for distance_name in distance_defs:
    pose_pivot[distance_name] = None

In [ ]:
for idx, row in pose_pivot.iterrows():

    coords = {
        name: (row[f'x{id}'], row[f'y{id}'], row[f'z{id}'])
        for id, name in joints
    }

    for distance_name, joint_def in distance_defs.items():

        if len(joint_def) == 2:
            a, b = joint_def

            p1 = coords[a]
            p2 = coords[b]

            pose_pivot.at[idx, distance_name] = \
                calculate_distance(p1, p2)

        else:
            a1, b1, a2, b2 = joint_def

            p1 = coords[a1]
            p2 = coords[b1]
            p3 = coords[a2]
            p4 = coords[b2]

            pose_pivot.at[idx, distance_name] = \
                calculate_distance4(p1, p2, p3, p4)

2.4 Scaling the data aorund the center of the body (hip)

In [ ]:
data = pose_pivot
data 

finding the middel of the body (middle of the hips)

In [ ]:
data['hip_x'] = (data['x23']+ data['x24'])/2
data['hip_y'] = (data['y23']+ data['y24'])/2
data['hip_z'] = (data['z23']+ data['z24'])/2
data


In [ ]:
for i in data:
    if "x" in i:
        data[i] = data[i] - data['hip_x']
    if "y" in i:
        data[i] = data[i] - data['hip_y']
    if "z" in i:
        data[i] = data[i] - data['hip_z']

